# Implementing `BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding` From Scratch

## Part 1: A Foundational Analysis of BERT

### Section 1: The Pre-BERT Landscape and the Need for Bidirectional Understanding

The central challenge in **Natural Language Processing (NLP)** has long been the development of models that can capture the rich, contextual meaning of human language. BERT was not an isolated invention but a direct and elegant solution to a well-defined set of limitations present in prior state-of-the-art models. <br>

**The Problem with Context-Free Embeddings:**<br>
Early successes in neural language modeling such as Word2Vec and GloVe revolutionized NLP by learning dense vector representations (embeddings) for words. These models operated on a simple principle: words that appear in similar contexts should have similar vector representations. This allowed models to capture semantic relationships, famously enabling vector arithmetic like:<br>
$king - man + woman ≈ queen.$<br>
But these embeddings were fundamentally static and context-free. The word "bank" for example, would have the exact same vector representation in "the river bank" and "the money in the bank." This inability to disambiguate words based on their surrounding texts a phenomenon known as **polysemy** posed a significant ceiling on the performance of downstream NLP tasks. The field needed representations that were dynamic and sensitive to context.<br>

Before BERT the dominant paradigm in NLP was to use language models that were unidirectional. Models like the canonical LSTMs or state-of-the-art GPT from OpenAI processed text sequentially either from left-to-right or right-to-left. <br>
Consider the sentence: *The man went to the bank to deposit money.* A left-to-right model would understand **bank** based only on *The man went to the...*. It couldn't use the crucial context **to deposit money** that comes after. This is a fundamental limitation because human language comprehension is not sequential it's holistic. We use the entire sentence to disambiguate meaning.

### Section-2: Assumptions, Goals, and Research Questions

- #### Primary Assumption
The paper assumes that a model pre-trained on a massive unlabeled text corpus can learn universal language representations that are beneficial for a wide variety of downstream NLP tasks (e.g., sentiment analysis, question answering). This builds upon prior work like Word2Vec and ELMo.

- #### Central Goal
To demonstrate that a deeply bidirectional model when pre-trained effectively will outperform unidirectional or shallowly bidirectional models across a wide range of NLP benchmarks and will establish a new state-of-the-art.

- #### Key Research Questions

1. Can the Transformer architecture originally designed for sequence-to-sequence tasks be adapted for language representation pre-training?
2. How can we train a truly bidirectional model given that seeing the whole sentence makes predicting the next word trivial?
3. Is learning inter-sentence relationships (like whether one sentence follows another) as important as learning intra-sentence relationships (word meanings)?
4. Can this pre-trained model be effectively fine-tuned for various tasks with minimal architectural changes, making it a general-purpose NLP backbone?

### 3. Methodology: Pre-training Tasks

We can't just feed a sentence to a bidirectional model and ask it to predict the next word, because it can already *see* the answer. There are two unsupervised pre-training tasks in the paper to solve this.

#### Task 1: Masked Language Model (MLM)

This is the core innovation that enables deep bidirectionality. It's inspired by the [Cloze task](https://www.teachingenglish.org.uk/professional-development/teachers/knowing-subject/c/cloze) which is a common reading comprehension test.

1. **Masking**: Before feeding a sentence to the model we randomly mask 15% of the tokens.
2. **The 80-10-10 Rule:** From the tokens we have chosen for masking:

    - 80% of the time the token is replaced with a special [MASK] token.
    - 10% of the time the token is replaced with a random token from the vocabulary.
    - 10% of the time the token is left unchanged.
3. **Objective:** The model's goal is to predict the original identity of the masked token based on the unmasked context from both left and right.

> Why the 80-10-10 split? 

It's an approach to reduce a mismatch between pre-training and fine-tuning. If we always use `[MASK]` the model would never learn good representations for actual words during pre-training and the `[MASK]` token would never appear during fine-tuning. By sometimes using a random word the model is forced to learn to correct it. By sometimes using the original word it biases the model towards the true representation.

**Example:**
- Original: The man went to the bank to deposit money.
- Masked Input: The man went to the `[MASK]` to `[MASK]` money.
- Model's Goal: Predict bank and deposit for the respective `[MASK]` positions.




#### Task 2: Next Sentence Prediction (NSP)

This task aims to teach the model to understand sentence relationships.

1. **Data Generation:** The training data is composed of sentence pairs (A, B).

    - 50% of the time sentence B is the actual sentence that follows sentence A in the corpus (labeled `IsNext`).
    - 50% of the time sentence B is a random sentence from the corpus (labeled `NotNext`).
2. **Objective:** The model must predict whether sentence B is the true next sentence for A. This prediction is made using the output of the special `[CLS]` token that is prepended to every input sequence.

**Example:**
- **Input A:** The man went to the bank.
- **Input B** (`IsNext`): He wanted to deposit his money. -> Label: `IsNext`
- **Input B** (`NotNext`): Penguins are flightless birds. -> Label: `NotNext`



## Part-2: Implementation from Scratch of BERT

### Section 1: Setup

In [6]:
import math
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

### Section-2: Configuration of Model

In [7]:
# class to hold parameters
class BERTConfig:
    # here 768 is the d_model in the original BERT paper
    # 12 is the number of layers in the original BERT paper(encoder blocks
    # 12 is the number of attention heads in the original BERT paper
    # 3072 is the intermediate size in the original BERT paper
    # 512 is the maximum position embeddings in the original BERT paper
    # 2 is the type vocab size in the original BERT paper
    # 0.1 is the dropout probability in the original BERT paper)
    def __init__(self,vocab_size=30, hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                 intermediate_size=3072, max_position_embeddings=512, type_vocab_size=2, hidden_dropout_prob=0.1):
        # initialising the BERT configuration parameters
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.attention_head_size = hidden_size // num_attention_heads
        self.intermediate_size = intermediate_size
        self.max_position_embeddings = max_position_embeddings
        self.type_vocab_size = type_vocab_size
        self.hidden_dropout_prob = hidden_dropout_prob


# using smaller dimensions for toy model
config = BERTConfig(vocab_size=30,hidden_size=128,num_hidden_layers=2,num_attention_heads=2,
    intermediate_size=128 * 4, max_position_embeddings=512, type_vocab_size=2, hidden_dropout_prob=0.1)

### Section-3: Building the BERT Components

- #### 3.1: Implementing Scaled Dot-Product Attention

Building the core of the attention mechanism:
```math
\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^\top}{\sqrt{d_k}} \right)V
```

Where:

* `Q` = Query matrix
* `K` = Key matrix
* `V` = Value matrix
* `d_k` = Dimension of the key vectors (used for scaling)
* `softmax` = Function to normalize attention scores into probabilities


In [8]:
class ScaledDotProductAttention(nn.Module):
    
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()# initialising the parent class
        
    def forward(self, Q,K,V, attention_mask):
        # Q, K, V are the query, key and value matrices with Q, K, V: [batch_size, n_heads, seq_len, head_dim]
        # attention mask is a mask to avoid attention on padding tokens with [batch_size, 1, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(Q.shape[-1]) # scaling the scores by the square root of the head dimension
        scores = scores + attention_mask# fill masked positions with a large negative value        
        attention_probs = nn.Softmax(dim = -1)(scores)  # softmax to get attention probabilities
        context = torch.matmul(attention_probs, V)  # context vector is the weighted sum of values for each token 
        
        # context shape: [batch_size, n_heads, seq_len, head_dim]
        return context

- #### 3.2: Multi-Head Attention

Implementing scaled dot-product atttention multiple times in parallel.

In [ ]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, config):
        super(MultiHeadAttention, self).__init__()# initialising the parent class
        self.n_heads = config.num_attention_heads# number of attention heads
        self.head_dim = config.attention_head_size# dimension of each attention head
        self.hidden_size = config.hidden_size# hidden size of the model from config
        
        # linear layers to project the input to query, key and value matrices
        self.W_Q = nn.Linear(self.hidden_size, self.n_heads * self.head_dim)# linear layer for queries
        self.W_K = nn.Linear(self.hidden_size, self.n_heads * self.head_dim)# linear layer for keys
        self.W_V = nn.Linear(self.hidden_size, self.n_heads * self.head_dim)# linear layer for values
        
        self.attention = ScaledDotProductAttention()# scaled dot product attention layer
        self.output_linear = nn.Linear(self.n_heads * self.head_dim, self.hidden_size)# linear layer to project the output back to hidden size
        self.dropout = nn.Dropout(config.hidden_dropout_prob)# dropout layer for regularization